# Visual Evaluation - Ministral 3.14B (No ReID)

In [1]:

import sys, os, sqlite3, json, subprocess, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / '.gitignore').exists():
        ROOT = p; break
sys.path.insert(0, str(ROOT / 'backend/src'))
os.environ['PROJECT_ROOT'] = str(ROOT)

env_path = ROOT / 'backend' / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

MODEL_LABEL = 'ministral_3_14b'
METHOD = 'no_reid'
METHOD_SUFFIX = f'_{METHOD}' if METHOD else ''
ABLATION_DIR = ROOT / 'data' / f'ablation_{MODEL_LABEL}{METHOD_SUFFIX}'
ANALYSIS_DIR = ROOT / 'data' / f'analysis_{MODEL_LABEL}{METHOD_SUFFIX}'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR = ROOT / 'data/videos/eval'
GT_PATH = ROOT / 'data/videos/eval/ground_truth.xlsx'

from service.impl.visual_service_impl import VisualServiceImpl
from service.impl.interval_service_impl import IntervalServiceImpl
from service.impl.events_service_impl import queries_for_condition
from utils.database import setup_database
from utils.vlm_client import VLMClient
from service.impl.config_store_service_impl import ConfigStoreServiceImpl as _CfgStore

_app_conn = sqlite3.connect(str(ROOT / 'data/analysis.db'))
_app_conn.row_factory = sqlite3.Row
_cfg_store = _CfgStore()
REL_VOCAB = _cfg_store.get_section(_app_conn, 'relation_vocab') or {}
_app_conn.close()
print(f'Project root: {ROOT}')
print(f'Method: no_reid / model: ministral_3_14b')


Project root: /home/ghiffaryr/iseql/multimodal-surveillance-iseql
Method: no_reid / model: ministral_3_14b


In [2]:

GRID_ROWS, GRID_COLS = 2, 4
VLM_DELAY = 0.1
MAX_RETRIES = 10
MEMORY_N = 3
MEMORY_TOP_K = 5
EMBED_PROVIDER = 'huggingface'
EMBED_MODEL = 'google/siglip-base-patch16-224'
PROVIDER = 'mistral'
MODEL = "ministral-14b-2512"

def detect_fps(video_path: str, default: int = 24) -> int:
    try:
        probe = subprocess.check_output(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=avg_frame_rate,r_frame_rate",
             "-of", "json", video_path], timeout=10, stderr=subprocess.DEVNULL)
        info = json.loads(probe)
        for key in ("avg_frame_rate", "r_frame_rate"):
            fps_str = info["streams"][0].get(key, "")
            if fps_str and "/" in fps_str:
                num, den = fps_str.split("/")
                fps = int(num) // int(den) if int(den) else 0
                if fps > 0:
                    return fps
    except Exception:
        pass
    return default

from service.impl.events_service_impl import default_deltas_for, derive_delta_fields
from service.impl.event_registry_service_impl import EventRegistryServiceImpl as _Reg

_evt_conn = sqlite3.connect(str(ROOT / 'data/analysis.db'))
_evt_conn.row_factory = sqlite3.Row
_reg = _Reg()
_DELTA_FIELDS = ('delta_visual', 'delta_audio', 'epsilon_visual', 'epsilon_audio',
                 'eta_visual', 'eta_audio', 'zeta_visual', 'zeta_audio', 'rho_visual', 'rho_audio')
DEFAULT_DELTAS = {}
for _cond in ('A', 'B', 'C'):
    for _e in _reg.list_events(_evt_conn, condition=_cond):
        _f = derive_delta_fields(_e.model_json)
        DEFAULT_DELTAS.update(default_deltas_for(_e.model_json, _e.id, _f))
_evt_conn.close()

def params_for_scene(scene) -> tuple[dict, int]:
    fps = detect_fps(str(VIDEO_DIR / f'scene{scene}.mp4'))
    def frames(d: dict) -> dict:
        return {
            k: (round(v * fps) if isinstance(v, (int, float)) and not isinstance(v, bool) else v)
            for k, v in d.items()
        }
    return frames(DEFAULT_DELTAS), fps


In [3]:

expected_df = pd.read_excel(GT_PATH, sheet_name='Expected Events')
expected_df = expected_df.dropna(subset=['scene'])
expected_df['scene'] = expected_df['scene'].astype(int)
expected_df['event'] = expected_df['event'].astype(str)

gt = pd.read_excel(GT_PATH, sheet_name='Ground Truth')
gt_visual = gt[gt['modality'] == 'visual'].dropna(subset=['scene'])
gt_visual['scene'] = gt_visual['scene'].astype(int)

print(f'Expected events: {{len(expected_df)}} rows, {{expected_df["scene"].nunique()}} scenes')
print('Events:', sorted(expected_df["event"].unique()))
print('Scenes:', sorted(expected_df["scene"].unique()))


Expected events: {len(expected_df)} rows, {expected_df["scene"].nunique()} scenes
Events: ['fight', 'gunshot_or_explosion', 'handoff', 'suspicious_near_vehicle', 'vehicle_collision', 'vehicle_escape']
Scenes: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]


In [4]:

db_path = ABLATION_DIR / f'{MODEL_LABEL}{METHOD_SUFFIX}.db'
if db_path.exists():
    db_path.unlink()
conn, cur = setup_database(db_path)
client = VLMClient(provider=PROVIDER, model=MODEL, temperature=0.0, seed=42)
visual = VisualServiceImpl(
    max_retries=MAX_RETRIES,
    relation_classids=REL_VOCAB.get('relation_classids') or [],
    relation_descriptions=REL_VOCAB.get('relation_descriptions') or {},
    memory_n=MEMORY_N,
    memory_top_k=MEMORY_TOP_K,
    embed_provider=EMBED_PROVIDER,
    embed_model=EMBED_MODEL,
)

for scene in sorted(expected_df['scene'].unique()):
    aid = f'{MODEL_LABEL}{METHOD_SUFFIX}_s{scene}'
    video = VIDEO_DIR / f'scene{scene}.mp4'
    if not video.exists():
        print(f'  Scene {scene}: video not found, skipping')
        continue
    fps = detect_fps(str(video))
    print(f'  Scene {scene}: fps={fps}, running {METHOD}...')
    visual.run_pipeline(
        video_path=str(video), conn=conn, client=client,
        grid_rows=GRID_ROWS, grid_cols=GRID_COLS,
        sampling_rate=fps, min_interval=VLM_DELAY,
        analysis_id=aid, track_objects=False,
        log=print,
    )
conn.close()
print('Pipeline done.')


  Scene 4: fps=24, running no_reid...
Starting video analysis: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/videos/eval/scene4.mp4
Provider: mistral, Model: ministral-14b-2512, Grid: 2x4
Sampling rate: 1 frame every 24 video frames
--- FRAME 0 ---
--- Frame 0 (mistral) ---
Calling mistral API (attempt 1)


  -> New person #1
  -> New person #2
  -> New person #3
  -> New person #4
  -> New person #5
  -> New person #6
  -> New object #7
  -> New object #8
  -> New object #9
  -> New object #10
  -> New object #11
  -> New object #12
  -> New object #13
Analyzing relations...
Rate limiter: waiting 0.10 s before next VLM call
Calling mistral API (attempt 1)


Relations: physical_altercation(1, 2)
  -> Saved relation physical_altercation(person) #1, Frame=0
  -> Saved relation physical_altercation(person) #2, Frame=0
--- FRAME 24 ---
--- Frame 24 (mistral) ---
Rate limiter: waiting 0.07 s before next VLM call
Calling mistral API (attempt 1)


  -> New person #14
  -> New person #15
  -> New person #16
  -> New object #17
  -> New object #18
  -> New object #19
  -> New object #20
  -> New object #21
  -> New object #22
  -> New object #23
Analyzing relations...
Rate limiter: waiting 0.10 s before next VLM call
Calling mistral API (attempt 1)


Relations: physical_altercation(14, 15)
  -> Saved relation physical_altercation(person) #15, Frame=24
  -> Saved relation physical_altercation(person) #14, Frame=24
--- FRAME 48 ---
--- Frame 48 (mistral) ---
Rate limiter: waiting 0.07 s before next VLM call
Calling mistral API (attempt 1)


...

Relations: none()
--- FRAME 216 ---
--- Frame 216 (mistral) ---
Rate limiter: waiting 0.08 s before next VLM call
Calling mistral API (attempt 1)


  -> New person #99
  -> New object #100
  -> New object #101
  -> New object #102
  -> New object #103
  -> New object #104
  -> New object #105
  -> New object #106
  -> New object #107
  -> New object #108
  -> New object #109
  -> New object #110
  -> New object #111
  -> New object #112
  -> New object #113
Analyzing relations...
Rate limiter: waiting 0.10 s before next VLM call
Calling mistral API (attempt 1)


Relations: carrying(99, backpack)
  -> Skipping hallucinated id 'backpack' in relation 'carrying'
  -> Saved relation carrying(person) #99, Frame=216
--- FRAME 240 ---
--- Frame 240 (mistral) ---
Rate limiter: waiting 0.08 s before next VLM call
Calling mistral API (attempt 1)


  -> New object #114
  -> New object #115
  -> New object #116
  -> New object #117
  -> New object #118
  -> New object #119
  -> New object #120
  -> New object #121
  -> New object #122
  -> New object #123
  -> New object #124
  -> New object #125
  -> New object #126
  -> New object #127
  -> New object #128
Analyzing relations...
Rate limiter: waiting 0.10 s before next VLM call
Calling mistral API (attempt 1)


Relations: none()
VLM analysis complete. 241 video frames processed.
Starting post-processing (sampling rate: 24)...
Loaded 15 vis-relation rows.
Computing duration intervals with merge logic...
Consolidated 9 unique relation intervals.
Filtered to 9 intervals with valid duration; saving...
Intervals saved successfully.
Interval construction complete.
Pipeline done.


In [5]:

conn = sqlite3.connect(str(db_path))
event_rows = []

for _, row in expected_df.iterrows():
    aid = f'{MODEL_LABEL}{METHOD_SUFFIX}_s{row["scene"]}'
    evt = row['event']
    params, fps = params_for_scene(row["scene"])
    sql_map = queries_for_condition("A", params, analysis_id=aid, fps=fps)
    sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
    df = pd.read_sql_query(sql, conn)
    det = not df.empty
    result = 'TP' if det else 'FN'

    vis_rels = ''
    if det:
        parts = []
        for _, r in df.iterrows():
            rel = evt
            sf = int(r['st'] * fps)
            ef = int(r['et'] * fps)
            parts.append(f'{rel}({sf}-{ef})')
        vis_rels = ', '.join(parts)
    else:
        all_rels = conn.execute(
            'SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?',
            (aid,)
        ).fetchall()
        if all_rels:
            parts = [f'{r}({sf}-{ef})' for r, sf, ef in all_rels]
            vis_rels = ', '.join(parts)

    event_rows.append({
        'scene': row['scene'], 'event': evt,
        'detected': 'YES' if det else 'NO', 'result': result,
        'relations': vis_rels,
    })

all_scenes = sorted(expected_df['scene'].unique())
for evt_fp in sorted(expected_df['event'].unique()):
    pos_scenes = set(expected_df[expected_df['event'] == evt_fp]['scene'])
    for neg_scene in all_scenes:
        if neg_scene in pos_scenes:
            continue
        aid = f'{MODEL_LABEL}{METHOD_SUFFIX}_s{neg_scene}'
        params, fps = params_for_scene(neg_scene)
        sql_map = queries_for_condition("A", params, analysis_id=aid, fps=fps)
        sql = sql_map.get(evt_fp, 'SELECT 0 WHERE 1=0')
        try:
            df = pd.read_sql_query(sql, conn)
            if not df.empty:
                rel_str = evt_fp + '(' + str(int(df.iloc[0]['st'] * fps)) + '-' + str(int(df.iloc[0]['et'] * fps)) + ')'
                event_rows.append({'scene': neg_scene, 'event': evt_fp,
                    'detected': 'YES', 'result': 'FP',
                    'relations': rel_str})
        except Exception:
            pass

conn.close()
edf = pd.DataFrame(event_rows)
edf['relations'] = edf['relations'].fillna('')

metrics = []
for evt in sorted(edf['event'].unique()):
    sub = edf[edf['event'] == evt]
    tpp = len(sub[sub['result'] == 'TP'])
    fpp = len(sub[sub['result'] == 'FP'])
    fnn = len(sub[sub['result'] == 'FN'])
    support = tpp + fnn
    p = tpp / (tpp + fpp) if (tpp + fpp) > 0 else 0.0
    r = tpp / (tpp + fnn) if (tpp + fnn) > 0 else 0.0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    metrics.append({
        'event': evt, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'TP': tpp, 'FP': fpp, 'FN': fnn, 'support': support
    })
metrics_df = pd.DataFrame(metrics)
print('\n=== Event Summary ===')
print(metrics_df.to_string(index=False))

with pd.ExcelWriter(ANALYSIS_DIR / f'visual_event_eval_{MODEL_LABEL}{METHOD_SUFFIX}.xlsx') as writer:
    metrics_df.to_excel(writer, sheet_name='Summary', index=False)
    for sc in sorted(expected_df['scene'].unique()):
        sc_df = edf[edf['scene'] == sc]
        if not sc_df.empty:
            sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)



=== Event Summary ===
                  event  precision  recall    f1  TP  FP  FN  support
                  fight      0.714     1.0 0.833   5   2   0        5
   gunshot_or_explosion      0.800     0.8 0.800   4   1   1        5
                handoff      0.000     0.0 0.000   0   0   5        5
suspicious_near_vehicle      0.000     0.0 0.000   0   0   5        5
      vehicle_collision      0.571     0.8 0.667   4   3   1        5
         vehicle_escape      0.000     0.0 0.000   0   0   5        5


In [6]:

tp = len(edf[edf['result'] == 'TP'])
fp = len(edf[edf['result'] == 'FP'])
fn = len(edf[edf['result'] == 'FN'])
support = tp + fn
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

conn2 = sqlite3.connect(str(db_path))
vpi = conn2.execute('SELECT COUNT(*) FROM VisualPerInterval').fetchone()[0]
conn2.close()

reid_flag = False if METHOD == 'no_reid' else True
result_df = pd.DataFrame([{
    'visual': MODEL_LABEL, 'reid': reid_flag,
    'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3),
    'TP': tp, 'FP': fp, 'FN': fn, 'support': support,
    'VPI': vpi,
}])
result_df.to_excel(ANALYSIS_DIR / 'summary.xlsx', index=False)
print(f'Summary: P={precision:.3f} R={recall:.3f} F1={f1:.3f} TP={tp} FP={fp} FN={fn} VPI={vpi}')


Summary: P=0.684 R=0.433 F1=0.531 TP=13 FP=6 FN=17 VPI=295


In [7]:

relation_types = {
    'physical_altercation',
    'running', 'enter_or_exit_vehicle', 'carrying', 
    'vehicle_collision', 'gunshot_visible',
    'explosion_visible',
}

conn = sqlite3.connect(str(db_path))

rel_rows = []
for scene in sorted(expected_df['scene'].unique()):
    aid = f'{MODEL_LABEL}{METHOD_SUFFIX}_s{scene}'
    scene_gts = gt_visual[gt_visual['scene'] == scene]
    if scene_gts.empty:
        continue

    cur = conn.execute(
        'SELECT DISTINCT RelationType FROM VisualRelation WHERE AnalysisID = ?',
        (aid,)
    )
    vlm_rels = {row[0] for row in cur.fetchall()}

    gt_rels = set(scene_gts['class'].unique())

    for rel in sorted(relation_types):
        in_gt = rel in gt_rels
        in_vlm = rel in vlm_rels
        if in_gt and in_vlm:
            result = 'TP'
        elif in_gt and not in_vlm:
            result = 'FN'
        elif not in_gt and in_vlm:
            result = 'FP'
        else:
            result = 'TN'
        rel_rows.append({
            'scene': scene, 'relation': rel,
            'in_gt': 'YES' if in_gt else 'NO',
            'in_vlm': 'YES' if in_vlm else 'NO',
            'result': result,
        })

conn.close()
rdf = pd.DataFrame(rel_rows)

rel_metrics = []
for rel in sorted(rdf['relation'].unique()):
    sub = rdf[rdf['relation'] == rel]
    tp = len(sub[sub['result'] == 'TP'])
    fn = len(sub[sub['result'] == 'FN'])
    fp = len(sub[sub['result'] == 'FP'])
    support = tp + fn
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    rel_metrics.append({
        'relation': rel, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'TP': tp, 'FP': fp, 'FN': fn, 'support': support
    })

rm_df = pd.DataFrame(rel_metrics)
print('\n=== Relation Summary ===')
print(rm_df.to_string(index=False))

with pd.ExcelWriter(ANALYSIS_DIR / f'visual_relation_eval_{MODEL_LABEL}{METHOD_SUFFIX}.xlsx') as writer:
    rm_df.to_excel(writer, sheet_name='Summary', index=False)
    for sc in sorted(expected_df['scene'].unique()):
        sc_df = rdf[(rdf['scene'] == sc) & (rdf['result'] != 'TN')]
        if not sc_df.empty:
            sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)

print(f'\nDone. XLSX written to {ANALYSIS_DIR}/')



=== Relation Summary ===
             relation  precision  recall    f1  TP  FP  FN  support
             carrying      0.357   1.000 0.526   5   9   0        5
enter_or_exit_vehicle      0.231   0.600 0.333   3  10   2        5
    explosion_visible      0.600   1.000 0.750   3   2   0        3
      gunshot_visible      1.000   0.333 0.500   1   0   2        3
 physical_altercation      0.714   1.000 0.833   5   2   0        5
              running      0.467   0.700 0.560   7   8   3       10
    vehicle_collision      0.571   0.800 0.667   4   3   1        5

Done. XLSX written to /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_ministral_3_14b_no_reid/
